In [6]:
import { createAgent, tool } from "npm:langchain";

import { ChatGoogle } from "npm:@langchain/google";

import { z } from "npm:zod";

import { parse } from "jsr:@std/dotenv";


In [7]:
const env = parse(await Deno.readTextFile(".env"));

const GOOGLE_API_KEY = env.GOOGLE_API_KEY;

const API_URL = env.RESTAURANT_API_URL;


In [8]:
const model = new ChatGoogle({
  model: "gemini-3.1-flash-lite",
  apiKey: GOOGLE_API_KEY,
  temperature: 0,
});


In [9]:
type RestaurantCategory =
  | "한식"
  | "일식"
  | "중식"
  | "양식"
  | "세계요리"
  | "특별한 술집"
  | "전통차/커피전문점"
  | "디저트/베이커리";

type Restaurant = {
  OPENDATA_ID: string;
  BZ_NM: string;
  GNG_CS: string;
  FD_CS: string;
  TLNO: string;
  MBZ_HR: string;
  SEAT_CNT: string;
  PKPL: string;
  HP: string;
  PSB_FRN: string;
  BKN_YN: string;
  INFN_FCL: string;
  BRFT_YN: string;
  DSSRT_YN: string;
  MNU: string;
  SMPL_DESC: string;
  SBW: string;
  BUS: string;
};

// 외부api 호출 함수
async function fetchRestaurants(district: string): Promise<Restaurant[]> {
  const url = new URL(API_URL);

  url.searchParams.set("addr", district);

  const response = await fetch(url);

  if (!response.ok) {
    throw new Error(`API 요청 실패: ${response.status}`);
  }

  const data = await response.json();

  // 실제 API 응답 구조에 맞게 수정
  return data.data as Restaurant[];
}

// 카테고리 필터 함수
function filterByCategory(
  restaurants: Restaurant[],
  category: RestaurantCategory | null,
): Restaurant[] {
  if (category === null) {
    return restaurants;
  }

  return restaurants.filter((restaurant) => restaurant.FD_CS === category);
}

// 지역,카테고리에 맞는 음식점 검색 함수
async function searchRestaurants(
  district: string,
  category: RestaurantCategory | null,
) {
  const restaurants = await fetchRestaurants(district);

  const filtered = filterByCategory(restaurants, category);

  return filtered;
}

const restaurantSearchTool = tool(
  // 실제로 실행할 함수
  async ({ district, category }) => {
    const restaurants = await searchRestaurants(district, category);

    return {
      count: restaurants.length,

      restaurants: restaurants.map((restaurant) => ({
        name: restaurant.BZ_NM,

        category: restaurant.FD_CS,

        address: restaurant.GNG_CS,

        menu: restaurant.MNU,

        parking: restaurant.PKPL,

        reservation: restaurant.BKN_YN,

        seats: restaurant.SEAT_CNT,

        description: restaurant.SMPL_DESC,
      })),
    };
  },
  // tool의 설명
  {
    name: "search_restaurants",

    description: `
대구광역시의 특정 지역에서
실제 음식점, 카페, 술집 정보를
공공데이터 API로 검색합니다.

사용자가 실제 음식점을 찾거나
추천을 요청할 때 사용합니다.

가격이 사용자의 예산에 맞는지
판정하는 용도로는 사용하지 마세요.
`,

    schema: z.object({
      district: z
        .string()
        .describe("검색할 대구광역시 행정구역. 예: 남구, 중구, 수성구"),

      category: z
        .enum([
          "한식",
          "일식",
          "중식",
          "양식",
          "세계요리",
          "특별한 술집",
          "전통차/커피전문점",
          "디저트/베이커리",
        ])
        .nullable()
        .describe("검색할 식당 카테고리. 특정 카테고리가 없으면 null"),
    }),
  },
);


예산판정도구 만들기
name, description, schema 파라미터를 통해 모델이 Tool의 용도와 입력을 이해하도록 정의한다

In [10]:
const budgetCheckTool = tool(
  async ({ restaurants, budgetMin, budgetMax }) => {
    const results = restaurants.map((restaurant) => {
      const matches = restaurant.menu.matchAll(
        /(\d{1,3}(?:,\d{3})+|\d+)\s*원/g,
      );

      const prices = Array.from(matches).map((match) =>
        Number(match[1].replaceAll(",", "")),
      );

      const matchedPrices = prices.filter(
        (price) => price >= budgetMin && price <= budgetMax,
      );

      return {
        name: restaurant.name,

        matched: matchedPrices.length > 0,

        matchedPrices,
      };
    });

    return {
      budgetMin,
      budgetMax,

      restaurants: results.filter((restaurant) => restaurant.matched),
    };
  },

  {
    name: "check_budget",

    description: `
search_restaurants Tool로 검색한 식당들의
메뉴 가격이 사용자의 1인당 예산 범위에
맞는지 확인합니다.

사용자가 가격이나 예산 조건을 말한 경우에만 사용하세요.

이 Tool을 사용하기 전에
먼저 search_restaurants를 사용하여
실제 식당 정보를 확보해야 합니다.
`,

    schema: z.object({
      restaurants: z
        .array(
          z.object({
            name: z.string(),

            menu: z.string(),
          }),
        )
        .describe("search_restaurants가 반환한 식당 이름과 메뉴 정보"),

      budgetMin: z.number().describe("사용자의 1인당 최소 예산"),

      budgetMax: z.number().describe("사용자의 1인당 최대 예산"),
    }),
  },
);


In [11]:
const agent = createAgent({
  model,

  tools: [restaurantSearchTool, budgetCheckTool],

  systemPrompt: `
당신은 대구 맛집 추천 AI 에이전트입니다.

실제 식당을 검색해야 하는 경우
search_restaurants Tool을 사용하세요.

사용자가 예산이나 가격 조건을 제시했다면:

1. 먼저 search_restaurants를 사용하여
   실제 식당을 검색합니다.

2. 그 결과 중 식당 이름과 메뉴 정보를 이용하여
   check_budget Tool을 사용합니다.

3. 예산에 맞는 식당만 최종적으로 추천합니다.

예산 조건이 없다면
check_budget Tool을 사용할 필요가 없습니다.

Tool에서 확인되지 않은
식당이나 가격을 만들어내지 마세요.
`,
});


In [14]:
// 예산검색 테스트
const result = await agent.invoke({
  messages: [
    {
      role: "user",

      content: "대구 남구에서 1인 3만원 정도로 먹을 수 있는 한식집 추천해줘",
    },
  ],
});

In [15]:
//tool 사용기록 확인
for (const message of result.messages) {
  if ("tool_calls" in message && message.tool_calls?.length) {
    console.log(message.tool_calls);
  }
}

[
  {
    type: "tool_call",
    id: "call_789845",
    name: "search_restaurants",
    args: { district: "남구", category: "한식" },
    thoughtSignature: "EnEKbwERTTIPKksRLWKwvcBmgWNIKr1GfPnXmssWroDdpErYQYuqYuC+GC2a0il1Sw6DUAgbo+SA5hPTWkok+Pw24Are8FAg5B1EGZAU+SMQpLXIKbu5dNmMl58ewJlFY66LUXqsW3rlShwLhNHiM04weQ=="
  }
]
[
  {
    type: "tool_call",
    id: "call_839578",
    name: "check_budget",
    args: {
      restaurants: [
        {
          menu: "다올상 22,000원 <br />해가빛상 34,000원 <br />모꼬지상 45,000원 <br />약선갈비찜 38,000원 <br />약선장어구이 33,000원 <br />약선모둠수육 33,000원<br />",
          name: "청라연"
        },
        {
          menu: "소갈비찜정식 20,000원 <br />소갈비찜+돌솥밥 22,000원 <br />능이돌솥밥정식10,000원 <br />뚝배기불고기정식11,000원 <br />차돌청국장정식 8,000원 <br />차돌된장정식 7,000원 <br />누룽지탕 7,000원<br />",
          name: "일미정"
        },
        {
          menu: "수선화정식 27,000원 <br />들국화정식 22,000원 <br />오리로스구이 20,000원 ~ 38,000원 <br />훈제오리 23,000원 ~ 45,000원<br /><br />",
          name: "물베기한정식"
        },
        {
   

In [17]:
const result = await agent.invoke({
  messages: [
    {
      role: "user",
      content: "대구 남구에서 1인 3만원 정도로 먹을 수 있는 한식집 추천해줘",
    },
  ],
});

console.log(result.messages.at(-1).content);


대구 남구에서 1인당 3만 원 정도의 예산으로 즐길 수 있는 한식집들을 추천해 드립니다.

*   **청라연**: 한정식 전문점으로, '다올상(22,000원)' 메뉴가 예산 범위 내에 있어 정갈한 한식을 즐기기에 좋습니다.
*   **일미정**: 약선 음식 전문점으로, '소갈비찜정식(20,000원)'이나 '소갈비찜+돌솥밥(22,000원)' 메뉴를 추천합니다.
*   **물베기한정식**: 다양한 정식 메뉴가 준비되어 있으며, '수선화정식(27,000원)', '들국화정식(22,000원)' 등이 예산에 적합합니다.
*   **백복수반**: 앞산 먹거리 마을에 위치한 곳으로, '불고기정식(20,000원)' 등을 예산 내에서 드실 수 있습니다.

방문하시기 전에 예약 가능 여부를 확인하시면 더욱 편리하게 이용하실 수 있습니다. 즐거운 식사 되세요!


Agent가

① 사용자의 목표를 이해하고

② 적절한 Tool을 선택하고

③ Tool 결과를 확인한 뒤

④ 다음 Tool이 필요한지 다시 판단한다.